In [1]:
import numpy as np
import pandas as pd

In [2]:
import nltk
#nltk.download('stopwords')
#nltk.download('wordnet')

In [3]:
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import re

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


In [9]:
df = pd.read_csv('moviereviews.tsv', sep='\t')

In [10]:
df.dropna(inplace=True)

In [11]:
df = df[df['review'].apply(lambda x: isinstance(x, str))]

In [12]:
df

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...
...,...,...
1995,pos,"i like movies with albert brooks , and i reall..."
1996,pos,it might surprise some to know that joel and e...
1997,pos,the verdict : spine-chilling drama from horror...
1998,pos,i want to correct what i wrote in a former ret...


In [13]:
from sklearn.model_selection import train_test_split

X = df['review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [14]:
def clean_text(text):
    text = str(text)
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.lower().split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)


In [15]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [16]:

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        #lowercase=None,
        stop_words=None,        
        ngram_range=(1,2),      
        max_df=0.9,
        min_df=5,
        preprocessor=clean_text
    )),
    ('clf', LinearSVC())
])


In [17]:
pipeline

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=5, ngram_range=(1, 2),
                                 preprocessor=<function clean_text at 0x000001F7F9B1C360>)),
                ('clf', LinearSVC())])

In [18]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)


In [19]:

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))


[[278  44]
 [ 49 278]]
              precision    recall  f1-score   support

         neg       0.85      0.86      0.86       322
         pos       0.86      0.85      0.86       327

    accuracy                           0.86       649
   macro avg       0.86      0.86      0.86       649
weighted avg       0.86      0.86      0.86       649

Accuracy: 0.8567026194144838
